# ThreatHunter QLoRA Fine-Tuning
Run this notebook in Google Colab with an A100 GPU.

In [ ]:
# ============================================================
# CELL 1 - Setup (Kaggle)
# ============================================================
!pip install transformers peft bitsandbytes trl accelerate datasets -q
import os
os.makedirs('/kaggle/working/huntgpt/threathunter_checkpoint', exist_ok=True)
os.makedirs('/kaggle/working/huntgpt/models/threathunter', exist_ok=True)
import urllib.request
print('Downloading training pairs...')
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/spoorthi2615/HuntGPT/main/data/training_pairs_real.json',
    '/kaggle/working/training_pairs_real.json'
)


In [ ]:
# ============================================================
# CELL 8 — Load base model with 4-bit quantization
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
)


In [ ]:
# ============================================================
# CELL 9 — Format training data as instruction pairs
# ============================================================
def format_example(pair):
    prompt = f"""Given this security log, identify the MITRE ATT&CK technique.
Respond ONLY with valid JSON: {{"technique_id": "T####.###", "technique_name": "...", "hypothesis": "..."}}
Log: {pair['log']}"""
    response = json.dumps({
        "technique_id": pair['technique_id'],
        "technique_name": pair['technique_name'],
        "hypothesis": f"Behavior consistent with {pair['technique_name']}."
    })
    return f"<s>[INST] {prompt} [/INST] {response}</s>"
with open('/kaggle/working/training_pairs_real.json') as f:
    data = json.load(f)
train_texts = [format_example(p) for p in data['train']]
val_texts = [format_example(p) for p in data['val']]
from datasets import Dataset
train_ds = Dataset.from_dict({"text": train_texts})
val_ds = Dataset.from_dict({"text": val_texts})
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")
print(train_texts[0])  # sanity check the format


In [ ]:
# ============================================================
# CELL 10 — LoRA config and SFTTrainer
# ============================================================
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
sft_config = SFTConfig(
    output_dir="/kaggle/working/huntgpt/threathunter_checkpoint",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="steps",
    eval_strategy="epoch",
    bf16=True,
    max_length=512,
    save_steps=50,
    dataset_text_field="text",
)
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
)
import os
from transformers.trainer_utils import get_last_checkpoint
last_checkpoint = None
if os.path.isdir("/kaggle/working/huntgpt/threathunter_checkpoint"):
    last_checkpoint = get_last_checkpoint("/kaggle/working/huntgpt/threathunter_checkpoint")
trainer.train(resume_from_checkpoint=last_checkpoint)


In [ ]:
# ============================================================
# CELL 11 — Save adapter to Drive
# ============================================================
trainer.model.save_pretrained('/kaggle/working/huntgpt/models/threathunter/')
tokenizer.save_pretrained('/kaggle/working/huntgpt/models/threathunter/')
print("Saved.")


In [ ]:
# ============================================================
# CELL 12 — Sanity check
# ============================================================
test_log = data['test'][0]
prompt = f"""Given this security log, identify the MITRE ATT&CK technique.
Respond ONLY with valid JSON: {{"technique_id": "T####.###", "technique_name": "...", "hypothesis": "..."}}
Log: {test_log['log']}"""
inputs = tokenizer(f"<s>[INST] {prompt} [/INST]", return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=150, temperature=0.1)
print("Expected technique:", test_log['technique_id'])
print("Model output:", tokenizer.decode(output[0], skip_special_tokens=True))
